In [ ]:
import json
import pandas as pd

# dataset - 1

In [ ]:
with open("normalized_products/dataset_1_normalized.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [ ]:
# rows, columns
print(df.shape)

(812, 58)


In [ ]:
print(df.columns)

Index(['id', 'brand', 'title', 'description', 'price', 'priceCurrency',
       'cluster_id', 'url', 'title_description', 'model', 'model_number',
       'product_type', 'chipset_name', 'chipset_series', 'vram',
       'storage_size', 'base_core_clock', 'boost_core_clock',
       'graphics_interface_speed', 'memory_clock', 'read_speed', 'write_speed',
       'random_4k_read_iops', 'random_4k_write_iops', 'bus_type',
       'memory_interface_width', 'interface_type', 'protocol',
       'max_resolution', 'max_monitor_support', 'video_out_interface',
       'core_shader_count', 'adaptive_sync', 'overclocked', 'directx_version',
       'opengl_version', 'width', 'length', 'height', 'weight', 'fan_count',
       'cooling_type', 'storage_connection_type', 'power_draw_w',
       'power_adapter', 'TBW_rating', 'MTBF_hours', 'operating_temperature',
       'manufacturing_process', 'memory_type', 'encryption_type', 'controller',
       'storage_technology', 'color', 'form_factor', 'blocked_slots'

In [ ]:
print(df.head())

         id            brand  \
0  12198483         Gigabyte   
1  78378158  Western Digital   
2  80641070          Corsair   
3  51886539          Seagate   
4  10328354             Asus   

                                               title  \
0  Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...   
1            WD Blue 6TB 3.5\" SATA 3 HDD/Hard Drive   
2                Corsair Force MP510 M.2 SSD - 960GB   
3         Seagate EXOS 4TB 3.5\" SATA HDD/Hard Drive   
4    Asus GeForce GTX 1650 Phoenix OC 4GB Video Card   

                                         description    price priceCurrency  \
0  CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...   799.99           GBP   
1  6TB WD Blue WD60EZAZ, 3.5\" HDD, SATA III - 6G...   129.98           GBP   
2  Solid State Drive, 960 GB, intern, M.2 2280, P...  2124.00           NOK   
3  4TB Seagate EXOS ST4000NM0115, 3.5\" Enterpris...   142.99           GBP   
4  The ASUS GeForce GTX 1650 Phoenix OC 4GB Video...   219.00           AUD

In [ ]:
df.isnull().sum().sort_values(ascending=False)

graphics_interface_speed    804
blocked_slots               801
adaptive_sync               799
power_adapter               799
TBW_rating                  797
opengl_version              794
power_draw_w                793
max_monitor_support         792
weight                      786
manufacturing_process       786
controller                  785
operating_temperature       785
case_material               784
directx_version             783
max_resolution              781
MTBF_hours                  775
width                       767
memory_clock                766
length                      765
encryption_type             763
cooling_type                762
fan_count                   761
height                      760
overclocked                 756
random_4k_read_iops         749
random_4k_write_iops        748
base_core_clock             733
video_out_interface         728
delivery_scope              727
memory_interface_width      727
protocol                    715
core_sha

The most interesting attributes are the ones where some values exist but many are missing. Those are exactly where a RAG system can shine. Here are the ones that we shall care about: 

- model_number (478 missing → ~40% present)
- storage_connection_type (399 missing)
- form_factor (395 missing)
- interface_type (376 missing)
- storage_size (241 missing)
- bus_type (184 missing)
- vram (583 missing but still present for GPUs)

In [ ]:
df[df.cluster_id == 1002037]

,id,brand,title,description,price,priceCurrency,cluster_id,url,title_description,model,...,manufacturing_process,memory_type,encryption_type,controller,storage_technology,color,form_factor,blocked_slots,case_material,delivery_scope
0,12198483,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,"CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...",799.99,GBP,1002037,https://www.novatech.co.uk/products/gigabyte-n...,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,NVIDIA GeForce RTX 3080 Gaming OC,...,None,GDDR6X,None,None,None,None,None,NaN,None,None


In [ ]:
cluster = df[df.cluster_id == 1002037]
cluster[["title", "vram", "storage_size", "interface_type", "bus_type", "form_factor", "storage_connection_type", "model_number"]]

,title,vram,storage_size,interface_type,bus_type,form_factor,storage_connection_type,model_number
0,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,10.0,NaN,None,None,None,None,None


In [ ]:
df["cluster_id"].nunique()
df.groupby("cluster_id").size().describe()

count    812.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
dtype: float64

- That means every cluster contains only one row. Hence, each product appears only once in this file.
- So the clusters you expected (multiple offers for the same product) are not inside this single dataset.

In [ ]:
def load_json(path):
    with open(path) as f:
        return pd.DataFrame(json.load(f))

df1 = load_json("normalized_products/dataset_1_normalized.json")
df2 = load_json("normalized_products/dataset_2_normalized.json")
df3 = load_json("normalized_products/dataset_3_normalized.json")
df4 = load_json("normalized_products/dataset_4_normalized.json")

In [ ]:
# combine them all
all_df = pd.concat([df1, df2, df3, df4])

In [ ]:
print(all_df.shape)

(3012, 58)


In [ ]:
# checking clusters again:
all_df.groupby("cluster_id").size().describe()

count    812.000000
mean       3.709360
std        0.574219
min        2.000000
25%        4.000000
50%        4.000000
75%        4.000000
max        4.000000
dtype: float64

- That means each product appears in about 3–4 different datasets.
- This is perfect for retrieval-based repair. One row is missing a value, while another store listing contains it.

In [ ]:
# identify which attributes can actually be repaired using cluster information
attributes = [
    "model_number",
    "storage_size",
    "bus_type",
    "interface_type",
    "form_factor",
    "vram",
    "storage_connection_type"
]


for attr in attributes:
    repairable = 0
    missing = 0
    
    for cid, group in all_df.groupby("cluster_id"):
        if group[attr].notnull().any():
            missing_rows = group[group[attr].isnull()]
            repairable += len(missing_rows)
        missing += group[attr].isnull().sum()
    
    print("Attribute: ", attr, "Repairable: ", repairable, "Total missing: ", missing)

Attribute:  model_number Repairable:  1187 Total missing:  1801
Attribute:  storage_size Repairable:  31 Total missing:  912
Attribute:  bus_type Repairable:  623 Total missing:  726
Attribute:  interface_type Repairable:  318 Total missing:  1438
Attribute:  form_factor Repairable:  276 Total missing:  1485
Attribute:  vram Repairable:  17 Total missing:  2163
Attribute:  storage_connection_type Repairable:  241 Total missing:  1511


RAG approach can actually make a difference. Let’s unpack it:

- model_number → 1187 / 1801 → Huge portion of missing values can actually be retrieved from other rows in the same cluster. Excellent candidate.
- bus_type → 623 / 726 → Also very promising.
- storage_size → 31 / 912 → Only a tiny fraction can be recovered. Might not be worth spending time here.
- interface_type → 318 / 1438 → Some potential.
- form_factor → 276 / 1485 → Moderate.
- vram → 17 / 2163 → Almost nothing to gain. Skip.
- storage_connection_type → 241 / 1511 → Some potential, but low.

For your first experiments, focus on model_number and bus_type. They have enough missing-but-recoverable values to generate meaningful results.

In [1]:
import pandas as pd

df1 = pd.read_json('normalized_products/dataset_1_normalized.json')
df2 = pd.read_json('normalized_products/dataset_2_normalized.json')
df3 = pd.read_json('normalized_products/dataset_3_normalized.json')
df4 = pd.read_json('normalized_products/dataset_4_normalized.json')
kb = pd.concat([df2, df3, df4], ignore_index=True)

target_attributes = ['model_number', 'bus_type', 'interface_type', 'form_factor', 'storage_connection_type']

print('=== DATASET 1 (to clean) ===')
print(f'Total rows: {len(df1)}')
print(f'Missing values per attribute:')
print(df1[target_attributes].isnull().sum())

print()
print('=== KNOWLEDGE BASE (datasets 2+3+4) ===')
print(f'Total rows: {len(kb)}')
print(f'Available values per attribute:')
print(kb[target_attributes].notna().sum())

print()
print('=== GROUND TRUTH RECOVERABLE VIA CLUSTER ID ===')
def get_ground_truth(cluster_id, attribute, kb):
    matches = kb[kb['cluster_id'] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {'', 'none', 'nan'}:
            return str(val).strip()
    return None

for attr in target_attributes:
    missing_rows = df1[df1[attr].isna()]
    recoverable = sum(1 for _, row in missing_rows.iterrows() if get_ground_truth(row['cluster_id'], attr, kb) is not None)
    print(f'{attr}: {len(missing_rows)} missing, {recoverable} recoverable ({100*recoverable/max(len(missing_rows),1):.0f}%)')

=== DATASET 1 (to clean) ===
Total rows: 812
Missing values per attribute:
model_number               478
bus_type                   184
interface_type             376
form_factor                395
storage_connection_type    399
dtype: int64

=== KNOWLEDGE BASE (datasets 2+3+4) ===
Total rows: 2200
Available values per attribute:
model_number                877
bus_type                   1658
interface_type             1138
form_factor                1110
storage_connection_type    1088
dtype: int64

=== GROUND TRUTH RECOVERABLE VIA CLUSTER ID ===
model_number: 478 missing, 302 recoverable (63%)
bus_type: 184 missing, 154 recoverable (84%)
interface_type: 376 missing, 64 recoverable (17%)
form_factor: 395 missing, 61 recoverable (15%)
storage_connection_type: 399 missing, 46 recoverable (12%)


In [4]:
import pandas as pd, requests

df1 = pd.read_json('normalized_products/dataset_1_normalized.json')
df2 = pd.read_json('normalized_products/dataset_2_normalized.json')
df3 = pd.read_json('normalized_products/dataset_3_normalized.json')
df4 = pd.read_json('normalized_products/dataset_4_normalized.json')
kb = pd.concat([df2, df3, df4], ignore_index=True)

# Get 5 rows where bus_type is missing and ground truth exists
def get_gt(cluster_id, attr, kb):
    matches = kb[kb['cluster_id'] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attr)
        if pd.notna(val) and str(val).strip().lower() not in {'', 'none', 'nan'}:
            return str(val).strip()
    return None

samples = []
for idx, row in df1.iterrows():
    if pd.isna(row.get('bus_type')):
        gt = get_gt(row['cluster_id'], 'bus_type', kb)
        if gt:
            samples.append((row, gt))
    if len(samples) == 5:
        break

for row, gt in samples:
    prompt = (
        "You are a product data expert. Fill in the missing attribute.\n\n"
        f"PRODUCT: title={row['title']}, brand={row.get('brand','')}\n\n"
        "MISSING ATTRIBUTE: bus_type\n\n"
        "Respond with VALUE:<answer> only. Example: VALUE:PCIe"
    )
    resp = requests.post("http://localhost:11434/api/generate",
        json={"model": "llama3.1:8b", "prompt": prompt, "stream": False})
    raw = resp.json()["response"].strip()
    print(f"GT: {gt:15} | RAW: {repr(raw)}")

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))